## Step 0 
Loading the libraries and defines small helper functions.

In [1]:
"Assessment 2 : Hotel Booking analysis pipeline. Usage:hotel_bookings.csv from. Produce: Number Table(CSV formate), PNG , log.txt. Requires: pandas numpy scipy scikit-learn statsmodels matplotlib "
import sys,os,warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix,roc_curve,precision_recall_curve,r2_score,mean_squared_error,mean_absolute_error)
warnings.filterwarnings("ignore")

SRC="hotel_bookings.csv" #original dataset taken from kaggel
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
SEED=1
LOG=open(os.path.join(OUT,"analysis_log.txt"),"w")

def log(*a):
    s="".join(str(x) for x in a)  
    print(s); LOG.write(s + "\n")
    
def table(df,name, index=True):
    df.to_csv(os.path.join(OUT, name), index=index)
    
def fig(name):
    plt.tight_layout(); plt.savefig(os.path.join(OUT,name),dpi=200);plt.sho()
    
MONTHS = ["January","February","March","April","May","June","July","August",
          "September","October","November","December"]
MNUM={m:i+1 for i,m in enumerate(MONTHS)}



## Step 1- Clean the data
**Objective:** Removing exact duplicates, zero-guest bookings,zero-night stays, negative or extreme ADR (average daily rate), and undefined values.
**Log**: Log are visible at log file for removed rows
**Quoted in report**: The counts are reported in assessment.

== 1. DATA CLEANING==
Raw rows/cols:(119390, 32)
Raw cancellation rate: 0.370
                                      Step   Rows  Removed
                                  Raw rows 119390        0
        After removing zero-guest bookings  87230    32160
           After removing zero-night stays  86639      591
           After removing exact duplicates  86637        2
             After removing undefined meal  86152      485
After removing undefined segment / channel  86147        5
Cleaned cancellation rate: 0.277
ADR==0 rows kept for Problem 1, excluded from Problem 2:1029
Clean analysis file:86147rows,20columns


In [4]:
#-----------------------------------------------------------1. CLEAN
raw=pd.read_csv(SRC)
log("== 1. DATA CLEANING==")
log("Raw rows/cols:", raw.shape)
log("Raw cancellation rate: %.3f" % raw.is_canceled.mean())
df = raw.drop_duplicates()
steps=[("Raw rows", len(raw))]
df= df.copy(); df["children"]=df["children"].fillna(0)
guests=df.adults + df.children + df.babies
nights=df.stays_in_weekend_nights + df.stays_in_week_nights
df=df[guests > 0]; steps.append(("After removing zero-guest bookings",len(df)))
nights=df.stays_in_weekend_nights + df.stays_in_week_nights
df =df[nights > 0];
steps.append(("After removing zero-night stays", len(df)))
df=df[(df.adr >=0) & (df.adr <=1000)]; steps.append(("After removing exact duplicates", len(df)))
df = df[df.meal != "Undefined"];    steps.append(("After removing undefined meal", len(df)))
df=df[(df.market_segment !="Undefined") & (df.distribution_channel !="Undefined")]
steps.append(("After removing undefined segment / channel",len(df)))
clean_log=pd.DataFrame(steps, columns=["Step","Rows"])
clean_log["Removed"]=(-clean_log.Rows.diff()).fillna(0).astype(int)
table(clean_log, "table_01_cleaning_log.csv", index=False)
log(clean_log.to_string(index=False))
log("Cleaned cancellation rate: %.3f" % df.is_canceled.mean())
log("ADR==0 rows kept for Problem 1, excluded from Problem 2:",(df.adr==0).sum())

COLS = ["is_canceled","adr","hotel","lead_time","arrival_date_year","arrival_date_month","stays_in_weekend_nights","stays_in_week_nights","adults","children","babies","meal", "market_segment", "distribution_channel", "is_repeated_guest","previous_cancellations", "deposit_type", "customer_type", "reserved_room_type","total_of_special_requests"]
d = df[COLS].copy()
d["month_num"] = d.arrival_date_month.map(MNUM)
d["total_nights"] = d.stays_in_weekend_nights + d.stays_in_week_nights
d.to_csv(os.path.join(OUT, "hotel_clean_20cols.csv"), index=False)
log("Clean analysis file:", d.shape[0], "rows,", len(COLS), "columns")


== 1. DATA CLEANING==
Raw rows/cols:(119390, 32)
Raw cancellation rate: 0.370
                                      Step   Rows  Removed
                                  Raw rows 119390        0
        After removing zero-guest bookings  87230    32160
           After removing zero-night stays  86639      591
           After removing exact duplicates  86637        2
             After removing undefined meal  86152      485
After removing undefined segment / channel  86147        5
Cleaned cancellation rate: 0.277
ADR==0 rows kept for Problem 1, excluded from Problem 2:1029
Clean analysis file:86147rows,20columns


## Step 2 - Descriptive analytics

**What it does:** Summary statistics for the numeric variables, cancellation rate by hotel,segment,deposit type and lead time, and median ADR by month and hotel.

**Observed:** Observing Figure 1-3 and the cancellation-rate tables. Groups cancel the most and ADR changes with season.

**Report:** The figures are explained in the report with respect to what it shows and why it matters to managers.

In [13]:
#---------DESCRIPTIVE
log("\n== 2. DESCRIPTIVE ANALYTICS ==")
num=["lead_time,""stay_in_weekend_nights","stays_in_week_nights","adults","children","babes","previous_cancellations","total_of_special_requests","adr"]
num = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "total_of_special_requests",
    "adr",
]

desc = d[num].describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
].round(2)
table(desc,"table_02_numeric_summary.csv");
log(desc.to_string())

def rate(col):
    t=d.groupby(col).is_canceled.agg(Bookings="size",Cancel_rate="mean").round(3)
    return t

cancel_tables = {c: rate(c) for c in ["hotel", "market_segment", "deposit_type", "customer_type",
                                      "distribution_channel", "meal", "arrival_date_year"]}
for c, t in cancel_tables.items():
        table(t, f"table_03_cancel_rate_by_{c}.csv"); log("\nCancellation rate by", c); log(t.to_string())
        
        #Fig 1 cancellation by segment & deposit
        fig_, ax=plt.subplots(1,2,figsize=(11,4))
for a,c in zip(ax,["market_segment","deposit_type"]):
            t=cancel_tables[c]["Cancel_rate"].sort_values()
            a.barh(t.index, t.values * 100, color="#2b6cb0")
            a.set_xlabel("Cancellatioin rate (%)");
            a.set_title(f"Cancellation rate by {c.replace('_','')}")
def fig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, name), dpi=200)
    plt.close()
        
        # Fig 2 lead time vs cancellation
d["lead_band"]=pd.cut(d.lead_time, [-1,7,30,90,180,365,800],labels=["0-7", "8-30", "31-90", "91-180", "181-365", "366+"])
lb = d.groupby("lead_band", observed=True).is_canceled.mean() * 100
table(d.groupby("lead_band", observed=True).is_canceled.agg(Bookings="size", Cancel_rate="mean").round(3),"table_04_cancel_rate_by_lead_band.csv")
plt.figure(figsize=(6, 4)); plt.bar(lb.index.astype(str), lb.values, color="#2b6cb0")
plt.xlabel("Lead time (days)"); plt.ylabel("Cancellation rate (%)"); plt.title("Cancellation rate rises with lead time")
fig("figure_02_cancel_by_lead_time.png")               

 # Fig 3 ADR by month and hotel
adr = d[d.adr > 0].groupby(["month_num", "hotel"]).adr.median().unstack()
adr.index = [MONTHS[i - 1][:3] for i in adr.index]
table(adr.round(1), "table_05_median_adr_by_month_hotel.csv")
adr.plot(marker="o", figsize=(7, 4), color=["#2b6cb0", "#dd6b20"])
plt.ylabel("Median ADR"); plt.xlabel("Arrival month"); plt.title("Median ADR by month and hotel (ADR > 0)")
fig("figure_03_adr_by_month_hotel.png")
log("\nMedian ADR by month and hotel:\n", adr.round(0).to_string())           
                


== 2. DESCRIPTIVE ANALYTICS ==
                             mean    std  min   25%   50%     75%    max
lead_time                   80.24  86.10  0.0  12.0  50.0  125.00  709.0
stays_in_weekend_nights      1.01   1.03  0.0   0.0   1.0    2.00   19.0
stays_in_week_nights         2.63   2.03  0.0   1.0   2.0    4.00   50.0
adults                       1.88   0.62  0.0   2.0   2.0    2.00   55.0
children                     0.14   0.46  0.0   0.0   0.0    0.00   10.0
babies                       0.01   0.11  0.0   0.0   0.0    0.00   10.0
previous_cancellations       0.03   0.37  0.0   0.0   0.0    0.00   26.0
total_of_special_requests    0.70   0.83  0.0   0.0   1.0    1.00    5.0
adr                        107.19  51.18  0.0  73.0  99.0  134.35  510.0

Cancellation rate byhotel
              Bookings  Cancel_rate
hotel                              
City Hotel       53038        0.302
Resort Hotel     33109        0.238

Cancellation rate bymarket_segment
                Bookings  Cance